In [1]:
from collections import defaultdict
corpus = [
    "best places to visit in india",
    "best places to visit in chennai",
    "best places to visit near me",
    "best places to visit during summer",
]
N = 5                 
D = 0.75               
UNK = "<UNK>"
word_freq = defaultdict(int)
for sentence in corpus:
    for word in sentence.lower().split():
        word_freq[word] += 1
vocab = set()
for word, freq in word_freq.items():
    if freq > 1:
        vocab.add(word)

vocab.add(UNK)
processed = []
for sentence in corpus:
    words = []
    for word in sentence.lower().split():
        if word in vocab:
            words.append(word)
        else:
            words.append(UNK)
    processed.append(words)
ngram_counts = {i: defaultdict(int) for i in range(1, N + 1)}
for words in processed:
    for n in range(1, N + 1):
        for i in range(len(words) - n + 1):
            gram = tuple(words[i:i+n])
            ngram_counts[n][gram] += 1
continuation = defaultdict(set)
for gram in ngram_counts[2]:
    continuation[gram[-1]].add(gram[:-1])
total_unique_bigrams = len(ngram_counts[2])
def kneser_ney(history, word):
    order = len(history) + 1
    if order == 1:
        return len(continuation[word]) / max(total_unique_bigrams, 1)
    history = tuple(history)
    ngram = history + (word,)
    count = ngram_counts[order].get(ngram, 0)
    history_count = ngram_counts[order-1].get(history, 0)
    if history_count == 0:
        return kneser_ney(history[1:], word)
    followers = set()
    for gram in ngram_counts[order]:
        if gram[:-1] == history:
            followers.add(gram[-1])
    first = max(count - D, 0) / history_count
    lamb = (D * len(followers)) / history_count
    backoff = kneser_ney(history[1:], word)
    return first + lamb * backoff
def predict(query, top_k=5):
    words = query.lower().split()
    words = [w if w in vocab else UNK for w in words]
    history = words[-4:]   
    scores = {}
    for word in vocab:
        scores[word] = kneser_ney(history, word)
    ranked = sorted(scores.items(),
                    key=lambda x: x[1],
                    reverse=True)
    return ranked[:top_k]
query = "best places to visit"
print("Query:", query)
print("\nAutocomplete Suggestions:\n")
for word, prob in predict(query):
    print(f"{word:12} Probability = {prob:.4f}")

Query: best places to visit

Autocomplete Suggestions:

<UNK>        Probability = 0.4986
in           Probability = 0.4929
to           Probability = 0.0028
places       Probability = 0.0028
visit        Probability = 0.0028
